# Lecture 01 — Motivation, Digital Image Fundamentals & Intensity Transforms

**Course:** Image Processing 

---

## Learning Objectives

By the end of this session you will be able to:

1. Load, inspect, and resize images with **scikit-image** and **NumPy**.
2. Work in multiple **colour spaces** (RGB, HSV, CIELAB, YCbCr) and interpret their channels.
3. Apply **point operations** (negative, gamma, logarithmic) and understand their LUTs.
4. Enhance contrast with **histogram equalization**, **CLAHE**, and **histogram matching**.
5. Implement histogram equalization **from scratch** using only NumPy.
6. Reason about **pixel neighbourhoods** and common **distance metrics**.

---

## How to use this notebook

* **Essential exercises (1.1–1.6):** required for all students.
* **Additional exercises (1.7–1.8):** for students who finish early or want a deeper challenge.
* Some exercises ask you to implement the algorithm yourself **before** (or instead of) calling a library function.
* Cells that start with `# YOUR CODE HERE` are where you should write your solution.

---

## Useful resources

### Matplotlib
**User guide**
https://matplotlib.org/stable/users/index.html

**API Reference**
https://matplotlib.org/stable/api/index.html#

**Cheat Sheet**
https://labex.io/cheatsheets/pdf/matplotlib-cheatsheet.pdf

### Scikit-Image

**Getting Started**
https://scikit-image.org/docs/stable/user_guide/numpy_images.html

**API Reference**
https://scikit-image.org/docs/stable/api/api.html

## Setup

Run the cell below once to install dependencies (already available on Colab).

In [ ]:
!pip install -q scikit-image matplotlib numpy


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from skimage import data, io, color, exposure, transform
from skimage.exposure import match_histograms
from skimage.util import img_as_float, img_as_ubyte

# Use the astronaut example image as our 'reference part' throughout
IMAGE_RGB = data.astronaut()  # 512×512×3, uint8


---
## Exercise 1.1 — Loading and Exploring the Reference Image

### Background

A **digital image** is an N-dimensional array of numbers. In scikit-image:

| Image type | NumPy shape | Typical dtype |
|------------|-------------|---------------|
| Grayscale  | (H, W)      | uint8 or float64 |
| RGB colour | (H, W, 3)   | uint8 or float64 |
| RGBA       | (H, W, 4)   | uint8 |

Two dtype conventions are common:
* **uint8** — integer pixels in [0, 255]. Compact; used for I/O.
* **float64** — real-valued pixels in [0.0, 1.0]. Preferred for computation. `skimage.util.img_as_float` converts for you.

`transform.resize(image, (H, W))` resamples to a new spatial resolution. The `anti_aliasing=True` flag applies a Gaussian blur before downsampling to prevent high-frequency aliasing artefacts.

### Tasks

1. Assign `IMAGE_RGB` (defined in the Setup cell) to a variable `image_original`.
2. Print its **dtype**, **shape**, and **value range** (min/max).
3. Display the image with `plt.imshow`. Turn off axis ticks.
4. Resize to **256×256** pixels with `transform.resize` (keep `anti_aliasing=True`).
5. Display the original and resized images **side by side**.
6. Repeat the resize with `order=0` (nearest-neighbour) and `order=3` (bicubic) at a very small size (e.g. 48×48) to see the difference. Display all three.

> **Hint:** `transform.resize` always returns a **float64** array in [0, 1], even if the input was uint8.

### Review Questions

1. What does the third dimension (`shape[2] == 3`) represent?
2. After calling `transform.resize`, why has the dtype changed from `uint8` to `float64`?
3. What visual artefact does `order=0` (nearest-neighbour) produce that `order=3` avoids?
4. Why is `anti_aliasing=True` important when **downsampling** but irrelevant when **upsampling**?

In [ ]:
# YOUR CODE HERE


---
## Exercise 1.2 — Colour Spaces and Channel Inspection 
### Background

Colour spaces provide different ways to encode the same visual information. Choosing the right space simplifies many tasks.

**RGB** — the default camera/display space. Channels are highly correlated (a change in brightness affects R, G, and B simultaneously).

**HSV (Hue–Saturation–Value)** — separates *chromatic content* (H, S) from *brightness* (V). Useful for colour-based selection and object detection under varying illumination. All three channels in [0, 1] after `color.rgb2hsv`.

**CIELAB (L\*a\*b\*)** — a perceptually uniform space modelled on human vision. L\* encodes lightness [0, 100]; a\* encodes the green↔red axis; b\* encodes the blue↔yellow axis. Uniform means that a fixed numerical distance ΔE corresponds to roughly the same perceived colour difference everywhere in the space. Use `color.rgb2lab` (input must be float [0,1]).

| Function | Input | Output range |
|---|---|---|
| `color.rgb2gray(img)` | float [0,1] | float [0,1], shape (H,W) |
| `color.rgb2hsv(img)`  | float [0,1] | float [0,1], shape (H,W,3) |
| `color.rgb2lab(img)`  | float [0,1] | L* [0,100], a*,b* [-128,127] |

### Tasks

1. Convert `image_working` (your 256×256 float image from Ex 1.1) to grayscale, HSV, and CIELAB using the functions above.
2. Display all **nine channels** (3 × RGB, 3 × HSV, 3 × Lab) in a 3×3 grid. Use informative colormaps (`'Reds_r'`, `'hsv'`, `'gray'`, etc.).
3. For the Lab image, normalise L\* to [0,1] and shift a\*, b\* so that 0 maps to the middle of the display range before calling `imshow`.
4. Identify which single channel/space offers the highest **contrast** between the brightest region (helmet) and the dark background. Justify your choice.

> **Hint:** `skimage.color` contains color transforms 

### Review Questions

1. Why do the RGB channels look so similar in overall brightness, while the HSV V channel looks almost identical to the grayscale image?
2. What does the a\* channel show when its value is close to its maximum vs. its minimum?
3. CIELAB is described as *perceptually uniform*. What practical consequence does this have for measuring colour similarity?
4. If you wanted to detect a bright yellow object against a dark background, which single channel would you choose and why?

In [ ]:
# YOUR CODE HERE


---
## Exercise 1.3 — Point Operations: Negative, Gamma, Logarithmic Transform 

### Background

A **point operation** (also called a *grey-level transform* or *intensity mapping*) maps each pixel independently: s = T(r), where r is the input intensity and s is the output intensity. The mapping T can be visualised as a **Look-Up Table (LUT)**.

| Transform | Formula | Effect |
|-----------|---------|--------|
| Negative  | s = 1 − r | Inverts bright↔dark; useful for X-rays |
| Gamma (γ < 1) | s = r^γ | Brightens dark regions (gamma correction) |
| Gamma (γ > 1) | s = r^γ | Darkens bright regions (exposure correction) |
| Logarithmic   | s = log₂(1+r) | Compresses wide dynamic ranges |

scikit-image provides:
* `exposure.adjust_gamma(img, gamma)` — gamma transform
* `exposure.adjust_log(img, gain=1)` — log transform (computes gain·log₂(1+img))

### Tasks

1. Start with `image_working` (float [0,1]).
2. Compute and display the following five images in a row:
   - Original
   - Negative
   - Gamma γ = 0.5
   - Gamma γ = 2.0
   - Logarithmic (`adjust_log`)
3. Below each image, plot its **intensity LUT**: a curve mapping input [0,1] → output [0,1].
   Use `np.linspace(0, 1, 256)` as the input axis.
4. Observe the effect of each transform on dark vs. bright pixels.

> **Hint:** `np.log1p(x) / np.log(2)` computes log₂(1+x) for an array x.

### Review Questions

1. For γ = 0.5, the LUT is concave (curves upward). What does this tell you about how dark pixels are treated compared to bright pixels?
2. Why would you apply a **logarithmic** transform to an image with a very bright specular reflection but a dark background?
3. The negative transform is its own inverse. Verify this algebraically.
4. What happens to the LUT shape when γ → 0? When γ → ∞?

In [ ]:
# YOUR CODE HERE


---
## Exercise 1.4 — Contrast Stretching and Histogram Equalization

### Background

The **image histogram** counts how many pixels have each intensity value. A narrow histogram means low contrast; a spread histogram means high contrast.

**Contrast stretching** linearly rescales pixel values so that the full output range [0, 1] is used. Using *percentile clipping* (e.g. map the 2nd–98th percentile to [0,1]) makes the stretch robust to bright/dark outliers.

```python
p2, p98 = np.percentile(gray, (2, 98))
stretched = exposure.rescale_intensity(gray, in_range=(p2, p98))
```

**Histogram equalization** goes further: it maps the intensity distribution through the image's own CDF so that the output histogram is approximately flat. It can reveal details hidden in shadows or highlights, but may amplify noise.

```python
equalized = exposure.equalize_hist(gray)
```

### Tasks

1. Convert `image_working` to grayscale.
2. Apply **contrast stretching** using the 2nd and 98th percentiles.
3. Apply **histogram equalization** with `equalize_hist`.
4. Display all three images in a row (original / stretched / equalized).
5. Below each image, plot its histogram (256 bins, range [0,1]).
6. Based on visual inspection, which method produces the clearest contrast between the bright helmet and the dark background?

> **Hint:** `img.ravel()` flattens a 2D array to 1D for use with `plt.hist`.

### Review Questions

1. After contrast stretching, is the histogram shape the same as the original? Does equalization change the shape?
2. What does the **CDF** of a perfectly equalized image look like?
3. Histogram equalization is a global operation. Can you think of a scenario where it would make an image *worse*?
4. Why is percentile-based clipping (2–98%) often better than min–max stretching?

In [ ]:
# YOUR CODE HERE


---
## Exercise 1.5 — Implement Histogram Equalization from Scratch

### Background

The histogram equalization algorithm has five steps:

1. **Compute the histogram** h(r): count pixels at each of the 256 intensity levels.
2. **Compute the CDF**: CDF(k) = Σ h(i) for i = 0 … k.
3. **Normalise**: divide CDF by the total number of pixels so CDF ∈ [0, 1].
4. **Build the LUT**: LUT[k] = round(CDF(k) × 255), mapping input level k to output level.
5. **Apply the LUT**: for each pixel with value r, the output is LUT[r].

The result is mathematically equivalent to what `skimage.exposure.equalize_hist` computes.

**NumPy tools you will need:**
* `np.histogram(arr, bins=256, range=(0, 255))` — returns (counts, bin_edges)
* `hist.cumsum()` — running total
* Integer array indexing: if `lut` is a 256-element array, `lut[img_u8]` applies it to every pixel in one vectorised step.

### Tasks

1. Convert the grayscale float image to uint8 (multiply by 255, cast with `.astype(np.uint8)`).
2. Compute the 256-bin histogram with `np.histogram`.
3. Compute and normalise the CDF.
4. Build the 256-entry LUT by scaling the CDF to [0, 255].
5. Apply the LUT to every pixel using integer array indexing.
6. Convert the result back to float [0,1] and display it alongside the library result.
7. Compute the **maximum absolute difference** between your result and `equalize_hist`. It should be < 0.02 (small differences are expected due to uint8 rounding).

> **Warning:** `np.histogram` returns `bin_edges` with **257 values** for 256 bins. Use only the 256-element `hist` array (the counts) for the CDF, not the bin edges.

### Review Questions

1. The CDF is a non-decreasing function. Why does this guarantee that the LUT preserves the relative ordering of intensities (bright pixels remain brighter)?
2. After equalization, is the output histogram exactly flat? Why or why not?
3. What would happen if two input levels have the same CDF value? Would they be mapped to the same output level?
4. If you apply histogram equalization twice, does the result change the second time? Explain why or why not.

In [ ]:
# YOUR CODE HERE — implement _manual_equalize(image_float) using only NumPy


def manual_equalize(image_float):
    """
    Pure NumPy histogram equalization.
    Input : float ndarray in [0, 1], shape (H, W)
    Output: (eq_float, lut) — equalized image in [0, 1] and the 256-entry LUT
    """
    # Step 1: quantise to uint8
    img_u8 = (np.clip(image_float, 0, 1) * 255).astype(np.uint8)

    # Step 2: compute histogram
    # YOUR CODE HERE

    # Step 3: compute and normalise CDF
    # YOUR CODE HERE

    # Step 4: build LUT
    # YOUR CODE HERE

    # Step 5: apply LUT
    # YOUR CODE HERE

    pass


# Test your implementation
# eq_manual, lut = manual_equalize(gray)
# eq_lib = exposure.equalize_hist(gray)
# diff = np.abs(eq_manual - eq_lib)
# print(f"Max difference: {diff.max():.4f}")


---
## Exercise 1.6 — CLAHE and Histogram Matching

### Background

**CLAHE (Contrast-Limited Adaptive Histogram Equalization)** is an improved version of histogram equalization designed for images with uneven illumination.
Instead of computing one global histogram, it:

1. Divides the image into small **tiles** (contextual regions).
2. Equalises each tile's histogram **independently**.
3. Clips the histogram at `clip_limit` **before** computing the CDF, which limits the amplification of noise in nearly uniform regions.
4. Uses **bilinear interpolation** between tile results to avoid block boundaries.

```python
clahe = exposure.equalize_adapthist(gray, clip_limit=0.03)
```

**Histogram matching** normalises the intensity distribution of a *source* image to match a *reference* image. This is useful for standardising images taken under different lighting conditions — a critical pre-processing step in industrial inspection.

```python
matched = match_histograms(source, reference)
```

### Tasks

1. Apply `equalize_adapthist` to the grayscale image. Try `clip_limit` values of 0.01, 0.03, and 0.10 and compare the results.
2. Load a second image with a noticeably different overall brightness (e.g. `data.camera()`, a darker grayscale image).
3. Resize the second image to match your working image size.
4. Use `match_histograms` to match the source image's histogram to the reference.
5. Display original / CLAHE / matched in a row with their histograms below.
6. Save the matched image with `skio.imsave('canonical_reference.png', img_as_ubyte(matched))`.

> **Hint:** `match_histograms` requires both images to have the same number of channels. For grayscale, pass 2D arrays directly (no `channel_axis` needed).

### Review Questions

1. What does `clip_limit` control? What happens visually as you increase it towards 1.0?
2. CLAHE is preferred over global equalization for medical images. Why?
3. Histogram matching is sometimes called *histogram specification*. Why is this a more accurate name?
4. After histogram matching, are the actual pixel values of the two images identical? What has been made equal?

In [ ]:
# YOUR CODE HERE


---
## Additional Exercises

The following exercises extend the material for students who want a deeper understanding. They are **not required**, but are strongly recommended if you have time.

---
## Exercise 1.7 — Pixel Neighbourhoods and Distance Metrics 🆕

### Background

Many image processing algorithms work with **pixel neighbourhoods** — the set of pixels considered 'adjacent' to a given pixel.

| Connectivity | Neighbours | Shared |
|---|---|---|
| **4-connected** | top, bottom, left, right | edge |
| **8-connected** | 4-connected + 4 diagonal | edge or corner |

For two pixels p = (r₁, c₁) and q = (r₂, c₂), three distance metrics are commonly used:

| Metric | Formula | Alias |
|--------|---------|-------|
| **City-block** | \|r₁−r₂\| + \|c₁−c₂\| | L₁, Manhattan |
| **Chessboard** | max(\|r₁−r₂\|, \|c₁−c₂\|) | L∞, Chebyshev |
| **Euclidean** | √((r₁−r₂)² + (c₁−c₂)²) | L₂ |

The choice of metric matters for algorithms like distance transforms, connected-component labelling, and morphological operations.

### Tasks

1. Place a **center pixel** at position (4, 4) on a 9×9 grid.
2. List all **4-connected** neighbours. Confirm there are exactly 4 for an interior pixel.
3. List all **8-connected** neighbours. Confirm there are exactly 8.
4. Choose a second pixel (e.g. (1, 7)) and compute all three distances to the center.
5. Create a visualisation showing the **unit ball** for each metric: highlight (in blue) every pixel whose distance from the center is ≤ 2. Plot all three side by side.

> **Hint:** `np.abs(a - b)` works element-wise on NumPy arrays, so `np.sum(np.abs(center - pixel))` gives city-block distance directly.

### Review Questions

1. The city-block unit ball is a **diamond** shape. Why?
2. The chessboard unit ball is a **square**. Why?
3. Under which metric is the 4-connected neighbourhood exactly the set of pixels within distance 1?
4. In a distance-transform algorithm that labels each pixel with its distance to the nearest background pixel, which metric would you choose to make diagonal and horizontal/vertical distances equal?

In [ ]:
# YOUR CODE HERE


---
## Exercise 1.8 — Colour-Space Micro-Survey 🆕

### Background

When comparing colour spaces for a detection task, a useful measure is the **Signal-to-Noise Ratio (SNR)** of a channel with respect to a labelled region of interest (ROI):

```
SNR = |mean(ROI) − mean(background)| / std(background)
```

A high SNR means the channel separates the region of interest from the background with little ambiguity — exactly what you want for thresholding.

You will convert the reference image to four colour spaces and tabulate the per-channel SNR over a manually defined region:

| Space | Channels | Function |
|-------|----------|----------|
| Grayscale | Gray | `color.rgb2gray` |
| HSV | H, S, V | `color.rgb2hsv` |
| CIELAB | L\*, a\*, b\* | `color.rgb2lab` |
| YCbCr | Y, Cb, Cr | `color.rgb2ycbcr` |

> **Note on YCbCr ranges:** `color.rgb2ycbcr` returns Y in [16, 235] and Cb/Cr in [16, 240]. Normalise before comparing SNR values across spaces.

### Tasks

1. Define a **custom region of interest** in the image: use a boolean mask to select a specific area (e.g. a rectangle that contains the bright helmet in the astronaut image).
2. Convert the float RGB image to all four colour spaces listed above.
3. For each channel, compute the SNR formula above using NumPy array operations.
4. Print a **summary table** with columns: Space | Channel | SNR.
5. Identify the channel with the highest SNR and explain why it performs best.
6. *Optional:* plot all channels in a grid with the region boundary overlaid.

### Review Questions

1. Why is the **L\*** channel of CIELAB a better measure of perceived brightness than the V channel of HSV?
2. If your defect region has a distinctive colour (e.g. rust on a metal surface), which channels in HSV and Lab would you expect to have the highest SNR?
3. Why might normalising channels before comparing SNR across different colour spaces be important?
4. The YCbCr space is widely used in image compression (JPEG). Why is it advantageous to store less information in the Cb and Cr channels?

In [ ]:
# YOUR CODE HERE
